In [27]:
import pandas as pd
import numpy as np

orders = pd.read_csv("olist_orders_dataset.csv")
items = pd.read_csv("olist_order_items_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")

print("Loaded")

Loaded


**MERGING DATA**

In [28]:
df = items.merge(orders, on="order_id")
df = df.merge(products, on="product_id")
df = df.merge(customers, on="customer_id")

print(df.shape)

(112650, 26)


**CLEANING DATA**

In [29]:
df = df[df["order_status"] == "delivered"]

df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])

df = df.dropna()

**CREATING FEATURES**

In [30]:
df["month"] = df["order_purchase_timestamp"].dt.to_period("M")

**FIXING REVENUE & PROFIT**

In [31]:
df["revenue"] = df["price"]
df["profit"] = df["price"] - df["freight_value"]

**CREATING ANALYTICAL TABLE**

In [32]:
base = df.groupby(
    ["product_id", "product_category_name", "month", "customer_state"]
).agg({
    "price": "mean",
    "order_item_id": "count",
    "freight_value": "mean",
    "revenue": "sum",
    "profit": "sum"
}).reset_index()

base.rename(columns={"order_item_id": "quantity"}, inplace=True)

print(base.head())

                         product_id  product_category_name    month  \
0  00066f42aeeb9f3007548bb9d3f33c38             perfumaria  2018-05   
1  00088930e925c41fd95ebfe695fd2655             automotivo  2017-12   
2  0009406fd7479715e4bef61dd91f2462        cama_mesa_banho  2017-12   
3  000b8f95fcb9e0096488278317764d19  utilidades_domesticas  2018-08   
4  000d9be29b5207b54e86aa1b1ac54872     relogios_presentes  2018-04   

  customer_state   price  quantity  freight_value  revenue  profit  
0             RS  101.65         1          18.59   101.65   83.06  
1             SP  129.90         1          13.93   129.90  115.97  
2             SP  229.00         1          13.10   229.00  215.90  
3             RS   58.90         2          19.60   117.80   78.60  
4             SP  199.00         1          19.27   199.00  179.73  


**ELASTICITY FOR ALL CATEGORIES**

In [33]:
import statsmodels.api as sm

elasticity_map = {}

for cat in base["product_category_name"].dropna().unique():
    data = base[base["product_category_name"] == cat].copy()

    data = data[data["quantity"] > 0]

    if len(data) > 10:
        data["log_price"] = np.log(data["price"])
        data["log_qty"] = np.log(data["quantity"])

        X = sm.add_constant(data["log_price"])
        model = sm.OLS(data["log_qty"], X).fit()

        elasticity_map[cat] = model.params[1]

base["elasticity"] = base["product_category_name"].map(elasticity_map)

print("Elasticity done")

/tmp/ipykernel_1672/1360002358.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  elasticity_map[cat] = model.params[1]
/tmp/ipykernel_1672/1360002358.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  elasticity_map[cat] = model.params[1]
/tmp/ipykernel_1672/1360002358.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  elasticity_map[cat] = model.params[1]
/tmp/ipykernel_1672/1360002358.py:17: FutureWarning: Series.__getitem_

Elasticity done


/tmp/ipykernel_1672/1360002358.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  elasticity_map[cat] = model.params[1]
/tmp/ipykernel_1672/1360002358.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  elasticity_map[cat] = model.params[1]


**ADDING NEW COLUMNS **

In [41]:
#  price simulation
price_change = 0.10

base["new_price"] = base["price"] * (1 + price_change)

base["new_qty"] = base["quantity"] * (
    1 + (base["elasticity"].fillna(-0.05) * price_change)
)

#  new profit
base["new_profit"] = (
    base["new_price"] * base["new_qty"]
) - (base["freight_value"] * base["new_qty"])

#  profit change
base["profit_change"] = base["new_profit"] - base["profit"]

#  percentage change
base["percentage_change"] = (
    (base["new_profit"] - base["profit"]) / base["profit"]
) * 100

#  decision logic
def pricing_decision(row):
    if row["elasticity"] < -0.05:
        return "Decrease Price"
    elif row["elasticity"] < -0.01:
        return "Keep Stable"
    else:
        return "Increase Price"

base["decision"] = base.apply(pricing_decision, axis=1)

#  freight analysis
base["freight_ratio"] = base["freight_value"] / base["price"]
base["freight_flag"] = base["freight_ratio"].apply(
    lambda x: "High Freight Cost" if x > 0.5 else "Normal"
)

print("All columns created successfully")

All columns created successfully


**ADDING A  DECISION SYSTEM**

In [42]:
def pricing_decision(row):
    if row["elasticity"] < -0.05:
        return "Decrease Price"
    elif row["elasticity"] < -0.01:
        return "Keep Stable"
    else:
        return "Increase Price"

base["decision"] = base.apply(pricing_decision, axis=1)

In [47]:
print(base["decision"].value_counts())

decision
Keep Stable       41262
Decrease Price    26742
Increase Price    13607
Name: count, dtype: int64


**FREIGHT ANALYSIS (RARE FEATURE)✌️✌️**

In [43]:
base["freight_ratio"] = base["freight_value"] / base["price"]

def freight_flag(x):
    if x > 0.5:
        return "High Freight Cost"
    else:
        return "Normal"

base["freight_flag"] = base["freight_ratio"].apply(freight_flag)

**PRICE**

In [44]:
price_change = 0.10

base["new_price"] = base["price"] * (1 + price_change)

base["new_qty"] = base["quantity"] * (
    1 + (base["elasticity"].fillna(-0.5) * price_change)
)

base["new_profit"] = (
    base["new_price"] * base["new_qty"]
) - (base["freight_value"] * base["new_qty"])

**PROFIT CHANGING**

In [45]:
base["profit_change"] = base["new_profit"] - base["profit"]

In [46]:
base.to_csv("final_pricing_data_v4.csv", index=False)
print("File ready for Tableau")

File ready for Tableau
